# Trabajo Práctico Integrador

## Port Log - Sprint 1

## Integrantes del grupo:

> Obtener el número de grupo de la pestaña del aula de "Trabajos Prácticos"
>
> Entrega UN SOLO integrante del grupo, debe ser siempre el mismo.
>
> Utilizar UN ÚNICO repositorio de github por grupo.

### Nombre y apellido
- Hector Oviedo
- Sofia Antonia Gamallo
- Alexander Seling
- Nicolas Hugo Diaz
- Ivan Agustin Sandiyu

## Repositorio GITHUB

Especificar en la siguiente línea cual es la url para acceder al repositorio del grupo.



https://github.com/HectorOviedoM/Port_Log-Grupo_16.git




## Consideraciones Generales e Importantes

- Desarrollo base: Los puntos solicitados son la base del trabajo práctico. Es posible la ampliación sin restricciones del mismo.
- Para el traceo se deben agregar 1 [secrets](https://drlee.io/how-to-use-secrets-in-google-colab-for-api-key-protection-a-guide-for-openai-huggingface-and-c1ec9e1277e0) para la utilización de github.
  - GITHUB_TOKEN
- Organización del Notebook:
  - Agregar antes del ejercicio 01 una celda donde se importan todas librerías necesarias.
  - Utilizar un título para cada celda y escribir cada funcionalidad en una celda con título.
- La defensa es individual.
- Los grupos deben mantenerse a lo largo del trabajo integrador.
- La opción `DESACTIVAR_GIT_PUSH` solo será utilizada por los profesores.

## Versionado y Trabajo en Equipo


- Control de Versiones: Utilizar herramientas de versionado
- Colaboradores: Brindar acceso al repositorio a las siguientes cuentas:
  - lcd-had141@ugr.edu.ar
  - fpasinato@ugr.edu.ar
- La descripción del commit debe ser con el siguiente formato `"Día X: \<comentario del commit\>"`
- Changelog: Cada ejercicio/punto del trabajo práctico es considerado como un día distinto de trabajo. Debe quedar constancia en un archivo CHANGELOG.md de los cambios generados cada día. El changelog debe escribirse de tal forma que lo primero que se observe es el último cambio realizado, de esta forma apenas se abre el archivo se pueden leer los últimos cambios realizados. La forma de escritura debe ser un encabezado entre corchetes [Ejercicio X] y luego debajo los cambios realizado como bullets.
- El archivo README.md debe contener el sprint en el que nos encontramos y debe contener:
  - El objetivo.
  - La introducción y el contexto del sprint de trabajo.

## Criterios de evaluación

- Funcionalidad: el código debe ejecutar sin errores al dar "Ejecutar todo" (Ctrl+F9). Validar con Runtime -> Restart session and run all antes de entregar.
- Claridad: código organizado, nombres descriptivos, comentarios solo donde corresponde (funciones, clases, métodos). No se admite `git add .`.
- Estilo: [type hints](https://docs.python.org/3/library/typing.html), [pep8](https://peps.python.org/pep-0008/).

- Herramientas: librerías vistas en la cursada. Sin gdrive ni herramientas alternativas.

## Condiciones de Entrega
- Archivo `.ipynb`, ejecutado y con datos precargados, sobre copia de la plantilla original manteniendo la nomenclatura del nombre.
- Canal: aula virtual. Correo a lcd-had141@ugr.edu.ar solo si el plazo finalizó. No se aceptan enlaces.
- Los trabajos que no cumplan con los requisitos no serán tomados en cuenta como nota final. Solo serán corregidos y evaluados con nota para la devolución correspondiente. Se considera incumplimiento a:
  - Entregas fuera de término.
  - Trabajos que no se entreguen sobre la plantilla de trabajo estipulada.
  - Trabajos que no se ejecuten completos (que contengan errores intermedios al dar "Ejecutar todo").

---



# Objetivo

Aplicar conocimientos de versionado, organización y análisis exploratorio de datos con pandas sobre un dataset real de operaciones portuarias.

## Introducción y Contexto del problema

### Sprint 1

El Puerto Fluvial de Rosario es uno de los complejos portuarios más importantes de América del Sur, siendo el principal punto de exportación de granos y derivados de la Argentina. Diariamente ingresan y egresan decenas de buques de distintas banderas con cargas de diverso tipo.

El sistema de registro de movimientos portuarios fue migrado recientemente desde un sistema heredado de los años '90. Ese sistema acumuló durante décadas inconsistencias de formato en fechas, matrículas de buques y valores numéricos fuera de rango, generando registros que no pueden incorporarse directamente al nuevo sistema.

Nuestro equipo fue contratado para analizar y depurar los datos del sistema antiguo.

Descargar el dataset [port_movements](https://raw.githubusercontent.com/HAD141/datasets/refs/heads/main/TrabajosPracticos/port_log/port_movements.csv)  

`https://raw.githubusercontent.com/HAD141/datasets/refs/heads/main/TrabajosPracticos/port_log/port_movements.csv`

---

In [ ]:
#@title Configurar variables de entorno y secretos de Colab
import os
from google.colab import userdata

HOOKS="/tmp/.git-templates/hooks"

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
try:
  DESACTIVAR_GIT_PUSH = bool(userdata.get('DESACTIVAR_GIT_PUSH'))
except userdata.SecretNotFoundError:
  DESACTIVAR_GIT_PUSH = False
os.environ['DESACTIVAR_GIT_PUSH'] = str(DESACTIVAR_GIT_PUSH)
os.environ['HOOKS'] = HOOKS

In [ ]:
#@title Crear el hook que bloquea los envíos durante la corrección
%%bash
mkdir -p $HOOKS
cat << EOF > $HOOKS/pre-push
#!/bin/sh
if [ "$DESACTIVAR_GIT_PUSH" = "'True'" ] || [ "$DESACTIVAR_GIT_PUSH" = "True" ]; then
    echo "[GIT HOOK - WARNING] Pushes are disabled for this repository."
    exit 1
fi
EOF

chmod +x $HOOKS/pre-push

In [ ]:
#@title Configurar el hook de Git para la corrección
if DESACTIVAR_GIT_PUSH:
  !git config --global core.hooksPath {HOOKS}

In [ ]:
#@title Importaciones

import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Ejercicio 01

> Puntos: 1

Inicialización y configuración de la herramienta de versionado sobre la rama `Sprint_1`.

La estructura de directorios es:

```
├── port_log
│   ├── data
│   │   ├── interim    (datasets procesados en pasos intermedios)
│   │   │   ├── plots
│   │   ├── processed  (datasets finales para otra aplicación)
│   │   ├── raw        (datasets en crudo)
│   ├── reports        (resúmenes estadísticos generados)
```

In [ ]:
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

In [ ]:
REPO_DIR = Path("/content/Port_Log-Grupo_16")
REPO_URL = f"https://github.com/HectorOviedoM/Port_Log-Grupo_16.git"
%cd /content
if REPO_DIR.exists():
    print("Ya existe el clone, actualizo...")
    %cd Port_Log-Grupo_16
    !git fetch origin
    !git checkout Sprint_1
    !git pull origin Sprint_1
else:
    !git clone {REPO_URL}
    %cd Port_Log-Grupo_16
    !git checkout Sprint_1
!pwd
!git status

In [ ]:
#@title Crear la estructura de carpetas
!mkdir -p port_log/data/raw
!mkdir -p port_log/data/interim/plots
!mkdir -p port_log/data/processed
!mkdir -p port_log/reports

In [ ]:
#@title Verificar la estructura de carpetas
!find port_log -type d | sort

In [ ]:
#@title Configurar el hook de Git para la corrección
if DESACTIVAR_GIT_PUSH:
  !git config --global core.hooksPath {HOOKS}

## Ejercicio 02

> Puntos: 1

- Descargar el dataset raw y almacenarlo en `port_log/data/raw/port_movements.csv`.


In [ ]:

#@title Ejercicio 02 — Descargar los datos originales
URL_RAW = "https://raw.githubusercontent.com/HAD141/datasets/refs/heads/main/TrabajosPracticos/port_log/port_movements.csv"
PATH_RAW = Path("port_log/data/raw/port_movements.csv")

def descargar_dataset_raw(url: str, destino: Path) -> pd.DataFrame:
    """Descarga el CSV, lo guarda en destino y lo retorna como DataFrame."""
    destino.parent.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(url)
    df.to_csv(destino, index=False)
    print(f"Guardado en {destino} — {df.shape[0]} filas x {df.shape[1]} columnas")
    return df

df_raw = descargar_dataset_raw(URL_RAW, PATH_RAW)

- Mostrar las 5 primeras y las 5 últimas filas en una única salida.


In [ ]:
#@title Primeras y últimas cinco filas
# Unir las primeras y las últimas cinco filas en una sola salida.
pd.concat([df_raw.head(5), df_raw.tail(5)])

- Analizar tipos de datos e indicar cuáles columnas requieren conversión.


In [ ]:
# Revisar los tipos de datos y los valores presentes.
df_raw.info()

A simple vista gracias a las funciones .info() y .head() se puede analizar que las principales columnas que requieren conversión son:

* `fecha_ingreso` y `fecha_egreso`: son de tipo texto (object) y tienen formatos inconsistentes o erróneos, por ejemplo, el registro 2 tiene 32/13/2021, un día y mes imposible. Se deben parsear a tipo fecha (datetime64) para detectar errores y realizar cálculos temporales.

* `hora_ingreso` y `hora_egreso`: también son de tipo texto y se mezclan formatos de 24 horas con formato AM/PM de 12 horas. Deben unificarse y convertirse a formato de tiempo normalizado de 24 horas.

* `matricula` y `muelle`: son de tipo texto, lo cual es correcto, pero necesitan limpieza de cadenas de texto.

In [ ]:
df_raw.describe()

La función *.describe()* muestra las estadísiticas generales de las columnas numéricas y se puede analizar que tanto `tonelaje_declarado` como `velocidad_ingreso` tienen valores atípicos muy extremos que necesitan ser transformados o limpiados.

- Contar los valores nulos y mostrarlos ordenados de mayor a menor.


In [ ]:
# Contar los nulos por columna, de mayor a menor.
nulos = df_raw.isna().sum().sort_values(ascending=False)

print(nulos)

- Calcular y mostrar el porcentaje de valores correctos por cada columna. Por ejemplo, si en una fila de fechas tengo una letra, entonces la contabilizo como valor incorrecto. Los valores nulos son considerados como valores incorrectos.

  Generar una única salida y debe tener el siguiente formato:
```
Completitud del dataset:
 - nombre_columna_1: XX.XX% completa
 - nombre_columna_2: XX.XX% completa
 - ...
```

In [ ]:
total_filas = len(df_raw)

# Validar las fechas.
def es_fecha_valida(val):
    if pd.isna(val):
        return False
    dt = pd.to_datetime(val, errors="coerce", format="mixed")
    return pd.notna(dt)

# Validar el formato de las horas.
def es_hora_valida(val):
    if pd.isna(val):
        return False
    val_str = str(val).strip()
    patron = r"^([0-1]?[0-9]|2[0-3]):[0-5][0-9](\s*([AP]\.?M\.?))?$"
    return bool(re.match(patron, val_str, re.IGNORECASE))

# Validar los campos de texto.
def es_texto_valido(val):
    if pd.isna(val):
        return False
    val_str = str(val).strip()
    return not bool(re.search(r"[@#$%!]", val_str))


# Definir una regla de validación para cada columna.
validaciones = {
    "movimiento_id": es_texto_valido,
    "buque_id": es_texto_valido,
    "matricula": es_texto_valido,
    "radar_id": es_texto_valido,
    "muelle": es_texto_valido,
    "tipo_carga": es_texto_valido,
    "origen": es_texto_valido,
    "fecha_ingreso": es_fecha_valida,
    "hora_ingreso": es_hora_valida,
    "fecha_egreso": es_fecha_valida,
    "hora_egreso": es_hora_valida,
    "tonelaje_declarado": lambda x: pd.notna(x),
    "velocidad_ingreso": lambda x: pd.notna(x),
    "velocidad_maxima_muelle": lambda x: pd.notna(x),
    "estado_despacho": es_texto_valido,
}

print("Completitud del dataset:")

for col in df_raw.columns:
    func_valida = validaciones.get(col, lambda x: pd.notna(x))
    correctos = df_raw[col].apply(func_valida).sum()
    porcentaje = (correctos / total_filas) * 100
    print(f" - {col}: {porcentaje:.2f}% completa")

## Ejercicio 03

> Puntos: 2

Utilizando Pandas, realizar la limpieza y normalización de datos:

- Normalizar fechas de ingreso y egreso al formato `YYYY-MM-DD`. Las fechas inválidas deben completarse con `1900-01-01`.

  Mostrar las 10 primeras fechas de ingreso.



In [ ]:
# Normalizar las fechas de ingreso.
df_clean = df_raw.copy(deep=True)

fecha_ing_dt = pd.to_datetime(df_clean['fecha_ingreso'], errors='coerce', format='mixed')
df_clean['fecha_ingreso'] = fecha_ing_dt.dt.strftime('%Y-%m-%d').fillna('1900-01-01')

# Normalizar las fechas de egreso.
fecha_egr_dt = pd.to_datetime(df_clean['fecha_egreso'], errors='coerce', format='mixed')
df_clean['fecha_egreso'] = fecha_egr_dt.dt.strftime('%Y-%m-%d').fillna('1900-01-01')

# Mostrar las primeras diez fechas de ingreso y egreso.
df_clean[['fecha_ingreso', 'fecha_egreso']].head(10)

- Normalizar las horas con el formato de 24hs. Las horas inválidas deben completarse con `00:00`.

  Mostrar los datos de las 10 primeras horas inválidas.


In [ ]:
# Convertir las horas al formato de 24 horas.
#@title Normalizar las horas e identificar los valores reemplazados
def normalizar_hora(val: object) -> str | None:
    """Normaliza una hora individual; devuelve None si es inválida."""
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    dt = pd.to_datetime(val_str, format="%H:%M", errors="coerce")
    if pd.isna(dt):
        dt = pd.to_datetime(val_str, format="%I:%M %p", errors="coerce")
    return dt.strftime("%H:%M") if pd.notna(dt) else None


# Identificar las horas inválidas antes de reemplazarlas.
invalidas_ingreso = pd.to_datetime(
    df_clean["hora_ingreso"], format="%H:%M", errors="coerce"
).isna() & pd.to_datetime(
    df_clean["hora_ingreso"], format="%I:%M %p", errors="coerce"
).isna()
invalidas_egreso = pd.to_datetime(
    df_clean["hora_egreso"], format="%H:%M", errors="coerce"
).isna() & pd.to_datetime(
    df_clean["hora_egreso"], format="%I:%M %p", errors="coerce"
).isna()

# Guardar en el CSV qué horas fueron reemplazadas.
df_clean["hora_ingreso_imputada"] = invalidas_ingreso
df_clean["hora_egreso_imputada"] = invalidas_egreso

# Mostrar los primeros diez registros con horas inválidas.
df_invalidas = df_clean[invalidas_ingreso | invalidas_egreso][
    ["hora_ingreso", "hora_egreso"]
].head(10)
print("10 primeras horas inválidas encontradas:")
display(df_invalidas)

# Normalizar las horas y reemplazar las inválidas por 00:00.
hora_ing_dt = pd.to_datetime(
    df_clean["hora_ingreso"], format="%H:%M", errors="coerce"
).fillna(
    pd.to_datetime(df_clean["hora_ingreso"], format="%I:%M %p", errors="coerce")
)
df_clean["hora_ingreso"] = hora_ing_dt.dt.strftime("%H:%M").fillna("00:00")

hora_egr_dt = pd.to_datetime(
    df_clean["hora_egreso"], format="%H:%M", errors="coerce"
).fillna(
    pd.to_datetime(df_clean["hora_egreso"], format="%I:%M %p", errors="coerce")
)
df_clean["hora_egreso"] = hora_egr_dt.dt.strftime("%H:%M").fillna("00:00")

# Los indicadores permiten distinguir un reemplazo de una medianoche válida.
print("Horas de ingreso imputadas:", int(invalidas_ingreso.sum()))
print("Horas de egreso imputadas:", int(invalidas_egreso.sum()))



- Calcular y agregar la columna `duracion_horas`: tiempo transcurrido entre `fecha_ingreso`+`hora_ingreso` y `fecha_egreso`+`hora_egreso`. Cuando el cálculo no sea posible (fechas inválidas en alguno de los extremos) asignar `pd.NA`.

  Mostrar los datos de las 10 primeras duraciones.


In [ ]:
# Calcular la duración; devolver pd.NA si las fechas son inválidas.
def calcular_duracion(row):
    if (
        row["fecha_ingreso"] == "1900-01-01"
        or row["fecha_egreso"] == "1900-01-01"
    ):
        return pd.NA

    try:
        # Combinar la fecha y la hora.
        inicio = pd.to_datetime(f"{row['fecha_ingreso']} {row['hora_ingreso']}")
        fin = pd.to_datetime(f"{row['fecha_egreso']} {row['hora_egreso']}")
        # Calcular la diferencia en horas.
        diferencia = (fin - inicio).total_seconds() / 3600.0
        # Si el egreso es anterior al ingreso, devolver pd.NA.
        return diferencia if diferencia >= 0 else pd.NA
    except Exception:
        return pd.NA

# Calcular la duración de cada movimiento.
df_clean["duracion_horas"] = df_clean.apply(calcular_duracion, axis=1)

# Mostrar las primeras diez duraciones.
df_clean[
    [
        "fecha_ingreso",
        "hora_ingreso",
        "fecha_egreso",
        "hora_egreso",
        "duracion_horas",
    ]
].head(10)


- Normalizar matrículas: quitar caracteres especiales y pasar a mayúsculas. Cuando la matrícula no sea válida completar con `pd.NA`.

  Mostrar los datos de las 10 últimas.


In [ ]:
def normalizar_matricula(val):
    if pd.isna(val):
        return pd.NA
    val_str = str(val).upper().strip()
    # Descartar las matrículas sin letras ni números.
    if not any(c.isalnum() for c in val_str):
        return pd.NA
    # Conservar letras, números, guiones y espacios.
    limpio = re.sub(r"[^A-Z0-9\-\s]", "", val_str).strip()
    return limpio if limpio else pd.NA

# Normalizar las matrículas.
df_clean["matricula"] = df_clean["matricula"].apply(normalizar_matricula)

# Mostrar las últimas diez matrículas.
df_clean[["matricula"]].tail(10)


- Normalizar muelles: quitar caracteres especiales y pasar a mayúsculas.

  Mostrar los datos de los 10 primeros.


In [ ]:
def normalizar_muelle(val):
    if pd.isna(val):
        return pd.NA
    # Pasar a mayúsculas y quitar los espacios de los extremos.
    val_str = str(val).upper().strip()
    # Quitar los caracteres especiales.
    limpio = re.sub(r"[^A-Z0-9\s\-]", "", val_str).strip()
    # Reemplazar los espacios repetidos por uno solo.
    limpio = re.sub(r"\s+", " ", limpio)
    return limpio if limpio else pd.NA

# Normalizar los nombres de los muelles.
df_clean["muelle"] = df_clean["muelle"].apply(normalizar_muelle)

# Mostrar los primeros diez muelles normalizados.
df_clean[["muelle"]].head(10)


- Eliminar filas con nulos en columnas críticas. Comentar el porque se seleccionaron esas columnas como críticas.

  Mostrar las columnas críticas y la cantidad de filas eliminadas para esa columna.
```
Las columnas críticas son:
 - nombre_columna: XXX
 - ...
```


In [ ]:
# Estas columnas identifican el movimiento y permiten analizar fechas, carga y velocidad.
columnas_criticas = [
    "movimiento_id",
    "matricula",
    "fecha_ingreso",
    "fecha_egreso",
    "hora_ingreso",
    "hora_egreso",
    "tonelaje_declarado",
    "velocidad_ingreso",
    "velocidad_maxima_muelle",
]

filas_antes = len(df_clean)
for columna in columnas_criticas:
    filas_antes_columna = len(df_clean)
    df_clean = df_clean.dropna(subset=[columna])
    filas_eliminadas = filas_antes_columna - len(df_clean)
    print(f"{columna}: {filas_eliminadas} filas eliminadas")

print("Columnas críticas:", columnas_criticas)


- Detectar y eliminar outliers en `tonelaje_declarado` y `velocidad_ingreso`. Para esto utilizaremos los métodos [IQR](https://www.geeksforgeeks.org/machine-learning/interquartile-range-to-detect-outliers-in-data/) y [Z-score](https://www.geeksforgeeks.org/data-science/z-score-in-statistics/).

  Comparar y justificar el método elegido en un comentario dentro de la celda.

  Mostrar las columnas críticas y la cantidad de filas eliminadas para esa columna.
```
Las columnas críticas son:
 - nombre_columna: XXX
 - ...
```

In [ ]:
# Organizar los gráficos en dos filas y dos columnas.
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Revisar los valores atípicos del tonelaje declarado.
sns.boxplot(data=df_clean, y='tonelaje_declarado', ax=axes[0, 0], color='lightblue',
            flierprops=dict(marker='o', markerfacecolor='red', markeredgecolor='darkred', alpha=0.9))
axes[0, 0].set_title("Boxplot de Tonelaje Declarado", fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel("Tonelaje")

# Observar la distribución del tonelaje.
sns.histplot(data=df_clean, x='tonelaje_declarado', ax=axes[0, 1], kde=True, color='steelblue', bins=50)
axes[0, 1].set_title("Distribución de Tonelaje Declarado", fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel("Tonelaje")
axes[0, 1].set_ylabel("Frecuencia")


# Revisar los valores atípicos de la velocidad de ingreso.
sns.boxplot(data=df_clean, y='velocidad_ingreso', ax=axes[1, 0], color='lightgreen',
            flierprops=dict(marker='o', markerfacecolor='red', markeredgecolor='darkred', alpha=0.9))
axes[1, 0].set_title("Boxplot de Velocidad de Ingreso", fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel("Velocidad de Ingreso")

# Observar la distribución de la velocidad.
sns.histplot(data=df_clean, x='velocidad_ingreso', ax=axes[1, 1], kde=True, color='seagreen', bins=50)
axes[1, 1].set_title("Distribución de Velocidad de Ingreso", fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel("Velocidad de Ingreso")
axes[1, 1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

In [ ]:
# Se observan valores extremos en ambas variables. Se elige IQR porque
# usa cuartiles y es menos sensible a esos valores que el Z-score,
# que depende de la media y la desviación estándar.

def eliminar_outliers_iqr(df, columna):
    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    filas_validas = df[columna].between(limite_inferior, limite_superior)
    return df[filas_validas].copy(), (~filas_validas).sum()

for columna in ["tonelaje_declarado", "velocidad_ingreso"]:
    df_clean[columna] = pd.to_numeric(df_clean[columna], errors="coerce")
    df_clean, filas_eliminadas = eliminar_outliers_iqr(df_clean, columna)
    print(f"{columna}: {filas_eliminadas} outliers eliminados")



- Crear columna `exceso_velocidad_real`: diferencia entre `velocidad_ingreso` y `velocidad_maxima_muelle`. Mostrar los datos de las 10 primeras matrículas con su valor.


In [ ]:
# Exceso real: diferencia entre la velocidad registrada y el límite del muelle.
df_clean["exceso_velocidad_real"] = (
    df_clean["velocidad_ingreso"] - df_clean["velocidad_maxima_muelle"]
)
print(df_clean[["matricula", "exceso_velocidad_real"]].head(10))


- Crear columna `exceso_velocidad`: diferencia entre `velocidad_ingreso` y `velocidad_maxima_muelle` más un 5% de tolerancia.

  Mostrar los datos de las 10 primeras matrículas con su valor.


In [ ]:
# Aplicar una tolerancia del 5% sobre la velocidad máxima permitida.
limite_con_tolerancia = df_clean["velocidad_maxima_muelle"] * 1.05
df_clean["exceso_velocidad"] = (
    df_clean["velocidad_ingreso"] - limite_con_tolerancia
)
print(df_clean[["matricula", "exceso_velocidad"]].head(10))


- Eliminar filas sin infracción según `exceso_velocidad`.

```
Se eliminaron X filas sin infracción.
```


In [ ]:
# Conservar los movimientos que superan el límite con tolerancia.
filas_antes = len(df_clean)
df_clean = df_clean[df_clean["exceso_velocidad"] > 0].copy()
print(f"Se eliminaron {filas_antes - len(df_clean)} filas sin infracción.")


- Guardar dataset limpio en `port_log/data/interim/port_movements.csv`.


In [ ]:
PATH_INTERIM = Path("port_log/data/interim/port_movements.csv")
PATH_INTERIM.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PATH_INTERIM, index=False)
print(f"Dataset limpio guardado en {PATH_INTERIM}")


- Exportar el resumen estadístico rápido de pandas en `port_log/reports/summary_sprint1.csv`.

In [ ]:
PATH_SUMMARY = Path("port_log/reports/summary_sprint1.csv")
df_clean.describe(include="all").to_csv(PATH_SUMMARY)
print(f"Resumen guardado en {PATH_SUMMARY}")

## Ejercicio 04

> Puntos: 2

Definir una clase `PortAnalyzer` que:

- Reciba el DataFrame limpio en `__init__` y encapsule los datos correctamente.

- Implemente los siguientes métodos:

  - **`top_infractores(n: int) -> pd.DataFrame`**: ranking de las `n` matrículas con más infracciones.
  
    Retorna un DataFrame ordenado de mayor a menor, índice desde 1, columnas `matricula` y `cantidad`.

  - **`infracciones_por_turno() -> pd.DataFrame`**: agrupa infracciones en turnos del día según la hora de ingreso:
    - Madrugada: 00:00–05:59
    - Mañana: 06:00–11:59
    - Tarde: 12:00–17:59
    - Noche: 18:00–23:59
    
    Retorna un DataFrame con columnas `turno` y `cantidad`, ordenado de mayor a menor.

  - **`exceso_promedio() -> float`**: Retorna el exceso de velocidad promedio calculado sobre la columna `exceso_velocidad_real`.

  - **`exceso_promedio_tolerancia() -> float`**: Retorna el exceso de velocidad promedio calculado sobre la columna `exceso_velocidad`.

  - **`infracciones_por_muelle() -> pd.DataFrame`**: Retorna un DataFrame con la cantidad de infracciones por muelle, ordenado alfabéticamente con las columnas `muelle` y `cantidad`.

  - **`infractores_por_tipo_carga() -> pd.DataFrame`**: Retorna un DataFrame con la cantidad de infracciones agrupadas por `tipo_carga`, de mayor a menor con las columnas `tipo_carga` y `cantidad`.



In [ ]:
#@title Ejercicio 04 — Clase PortAnalyzer
class PortAnalyzer:
    """Analiza movimientos ya filtrados como infractores sin mutar el original."""

    def __init__(self, df_infractores: pd.DataFrame) -> None:
        self._df_limpio = df_infractores.copy(deep=True)

    def top_infractores(self, n: int) -> pd.DataFrame:
        """Devuelve las n matrículas más reincidentes, con índice desde 1."""
        if n < 0:
            raise ValueError("n debe ser mayor o igual a cero.")
        resultado = (
            self._df_limpio.groupby("matricula")
            .size()
            .sort_values(ascending=False)
            .head(n)
            .reset_index(name="cantidad")
        )
        resultado.index = range(1, len(resultado) + 1)
        return resultado

    def infracciones_por_turno(self) -> pd.DataFrame:
        """Cuenta por turno según las horas normalizadas, de mayor a menor.

        Las horas imputadas a 00:00 se incluyen en Madrugada. Para interpretar
        esta distribución, el ejercicio 07 también muestra solo horas válidas.
        """
        turno = pd.cut(
            self._df_limpio["hora_ingreso"].str[:2].astype(int),
            bins=[0, 6, 12, 18, 24],
            labels=["Madrugada", "Mañana", "Tarde", "Noche"],
            right=False,
        )
        return (
            self._df_limpio.groupby(turno, observed=True)
            .size()
            .sort_values(ascending=False)
            .rename("cantidad")
            .rename_axis("turno")
            .reset_index()
        )

    def exceso_promedio(self) -> float:
        """Calcula el exceso real promedio; los valores ausentes se omiten."""
        return float(self._df_limpio["exceso_velocidad_real"].mean())

    def exceso_promedio_tolerancia(self) -> float:
        """Calcula el exceso promedio después de aplicar la tolerancia."""
        return float(self._df_limpio["exceso_velocidad"].mean())

    def infracciones_por_muelle(self) -> pd.DataFrame:
        """Cuenta infracciones por muelle, ordenadas alfabéticamente."""
        return (
            self._df_limpio.groupby("muelle")
            .size()
            .sort_index()
            .reset_index(name="cantidad")
        )

    def infractores_por_tipo_carga(self) -> pd.DataFrame:
        """Cuenta infracciones por tipo de carga, de mayor a menor."""
        return (
            self._df_limpio.groupby("tipo_carga")
            .size()
            .sort_values(ascending=False)
            .reset_index(name="cantidad")
        )


- Crear el objeto `PortAnalyzer` e invocar cada método en celdas separadas.

In [ ]:
analyzer = PortAnalyzer(df_clean)

In [ ]:
#@title Matrículas con más infracciones
analyzer.top_infractores(5)


In [ ]:
#@title Infracciones por turno
analyzer.infracciones_por_turno()

In [ ]:
#@title Ejercicio 04 — Exceso de velocidad real promedio
analyzer.exceso_promedio()

In [ ]:
#@title Ejercicio 04 — Exceso promedio con tolerancia
analyzer.exceso_promedio_tolerancia()

In [ ]:
#@title Ejercicio 04 — Cantidad de infracciones por muelle
analyzer.infracciones_por_muelle()

In [ ]:
#@title Ejercicio 04 — Infracciones por tipo de carga
analyzer.infractores_por_tipo_carga()

## Ejercicio 05

> Puntos: 2

Responder cada punto con un gráfico. Cada gráfico se debe mostrar en el notebook. Todos deben tener título, etiquetas en los ejes y leyenda cuando corresponda.

- **Top 10 matrículas más reincidentes** en barras verticales, de mayor a menor.

  Exportar: `port_log/data/interim/plots/top_infractores.jpg`.



In [ ]:
#@title Diez matrículas con más infracciones
top10 = analyzer.top_infractores(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top10, x="matricula", y="cantidad")
plt.title("Top 10 matrículas más reincidentes")
plt.xlabel("Matrícula")
plt.ylabel("Cantidad de infracciones")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("port_log/data/interim/plots/top_infractores.jpg", dpi=300, bbox_inches="tight")

- **Total de infracciones por turno del día** (Madrugada / Mañana / Tarde / Noche) en gráfico de torta.

  Exportar: `port_log/data/interim/plots/turnos.jpg`.


In [ ]:
#@title Total de infracciones por turno del día
turnos = analyzer.infracciones_por_turno()

plt.figure(figsize=(8, 8))
plt.pie(
    turnos["cantidad"],
    labels=None,
    autopct="%1.1f%%",
    startangle=90,
)
plt.title("Total de infracciones por turno del día")
plt.legend(turnos["turno"], title="Turno", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))
plt.tight_layout()
plt.savefig("port_log/data/interim/plots/turnos.jpg", dpi=300, bbox_inches="tight")


- **Total de infracciones por mes** en barras horizontal, ordenado de mayor a menor.

  Exportar: `port_log/data/interim/plots/meses.jpg`.


In [ ]:
#@title Total de infracciones por mes
MESES_ES = {
    "01": "Enero", "02": "Febrero", "03": "Marzo", "04": "Abril",
    "05": "Mayo", "06": "Junio", "07": "Julio", "08": "Agosto",
    "09": "Septiembre", "10": "Octubre", "11": "Noviembre", "12": "Diciembre",
}

meses = (
    df_clean.loc[df_clean["fecha_ingreso"] != "1900-01-01", "fecha_ingreso"]
    .str[:7]
    .value_counts()
    .sort_values(ascending=False)
)
etiquetas = [f"{MESES_ES[m.split('-')[1]]} {m.split('-')[0]}" for m in meses.index]

ALTO_BARRA = 0.5
alto = max(5, len(meses) * 0.55)

plt.figure(figsize=(10, alto))
plt.barh(etiquetas[::-1], meses.values[::-1], height=ALTO_BARRA, color="#4C72B0")
plt.title("Total de infracciones por mes")
plt.xlabel("Cantidad de infracciones")
plt.ylabel("Mes")
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig("port_log/data/interim/plots/meses.jpg", dpi=300, bbox_inches="tight")


- **Histograma del exceso de velocidad real** con curva de densidad (KDE) superpuesta.

  Exportar: `port_log/data/interim/plots/distribucion_exceso.jpg`.


In [ ]:
#@title Histograma del exceso de velocidad real con KDE
plt.figure(figsize=(10, 6))
sns.histplot(
    data=df_clean,
    x="exceso_velocidad_real",
    bins=30,
    stat="density",
    label="Histograma",
)
sns.kdeplot(
    data=df_clean,
    x="exceso_velocidad_real",
    label="Densidad (KDE)",
)
plt.title("Distribución del exceso de velocidad real")
plt.xlabel("Exceso de velocidad real (km/h)")
plt.ylabel("Densidad")
plt.legend()
plt.tight_layout()
plt.savefig("port_log/data/interim/plots/distribucion_exceso.jpg", dpi=300, bbox_inches="tight")


- **Exceso de velocidad promedio por muelle** en barras horizontal, ordenado de mayor a menor.

  Exportar: `port_log/data/interim/plots/exceso_por_muelle.jpg`.


In [ ]:
#@title Exceso de velocidad promedio por muelle
exceso_muelle = (
    df_clean.groupby("muelle")["exceso_velocidad_real"]
    .mean()
    .sort_values(ascending=False)
)

ALTO_BARRA = 0.6
alto = max(5, len(exceso_muelle) * 0.8)

plt.figure(figsize=(10, alto))
plt.barh(exceso_muelle.index[::-1], exceso_muelle.values[::-1], height=ALTO_BARRA, color="#55A868")
plt.title("Exceso de velocidad promedio por muelle")
plt.xlabel("Exceso promedio (km/h)")
plt.ylabel("Muelle")
plt.yticks(fontsize=11)
plt.tight_layout()
plt.savefig("port_log/data/interim/plots/exceso_por_muelle.jpg", dpi=300, bbox_inches="tight")


- **Comparación de infracciones con fecha válida vs fecha inválida (`1900-01-01`)** en barras simple.

  Exportar: `port_log/data/interim/plots/fechas_invalidas.jpg`.

In [ ]:
#@title Infracciones con fecha válida e inválida
n_invalidas = int((df_clean["fecha_ingreso"] == "1900-01-01").sum())
n_validas = int((df_clean["fecha_ingreso"] != "1900-01-01").sum())

categorias = ["Fecha válida", "Fecha inválida (1900-01-01)"]
valores = [n_validas, n_invalidas]

plt.figure(figsize=(8, 6))
barras = plt.bar(categorias, valores, width=0.5, color=["#4C72B0", "#C44E52"])
plt.title("Infracciones según validez de la fecha de ingreso")
plt.ylabel("Cantidad de infracciones")
for barra, valor in zip(barras, valores):
    plt.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + len(df_clean) * 0.005,
        str(valor),
        ha="center",
        fontsize=11,
    )
plt.tight_layout()
plt.savefig("port_log/data/interim/plots/fechas_invalidas.jpg", dpi=300, bbox_inches="tight")

## Ejercicio 06

> Puntos: 1

Responder las siguientes preguntas con el dataset limpio:

- ¿Qué porcentaje de infracciones provienen de registros con fecha inválida?
```
El porcentaje de infracciones con fecha inválida es XX.XX%
```



In [ ]:
#@title Ejercicio 06 — Porcentaje de infracciones con fecha inválida
# Contar una vez cada registro con fecha de ingreso o egreso inválida.
# El ejercicio 03 dejó en df_clean solo los registros infractores.
total_infracciones = len(df_clean)
if total_infracciones == 0:
    raise ValueError("El dataset limpio no contiene infracciones para analizar.")

fechas_infracciones = df_clean[["fecha_ingreso", "fecha_egreso"]]
fecha_invalida_6 = (
    fechas_infracciones.eq("1900-01-01") | fechas_infracciones.isna()
).any(axis=1)
porcentaje_fecha_invalida = 100 * fecha_invalida_6.sum() / total_infracciones
print(
    "El porcentaje de infracciones con fecha inválida es "
    f"{porcentaje_fecha_invalida:.2f}%"
)


- ¿Qué porcentaje de infracciones provienen de registros con hora inválida?
```
El porcentaje de infracciones con hora inválida es XX.XX%
```


In [ ]:
#@title Ejercicio 06 — Porcentaje de infracciones con hora inválida
def detectar_horas_invalidas(serie: pd.Series) -> pd.Series:
    """Detecta horas no interpretables con los dos formatos del ejercicio 03."""
    horas_24 = pd.to_datetime(serie, format="%H:%M", errors="coerce")
    horas_12 = pd.to_datetime(serie, format="%I:%M %p", errors="coerce")
    return horas_24.isna() & horas_12.isna()


# Verificar que los índices correspondan a los mismos movimientos
# para consultar las horas originales, antes de su reemplazo.
if (
    not df_raw.index.is_unique
    or not df_clean.index.is_unique
    or not df_clean.index.isin(df_raw.index).all()
):
    raise ValueError("No se puede vincular el dataset limpio con el original.")
originales_infractores = df_raw.loc[df_clean.index]
if not originales_infractores["movimiento_id"].equals(
    df_clean["movimiento_id"]
):
    raise ValueError("Los índices no corresponden a los mismos movimientos.")

# No contar 00:00 como error: puede ser una medianoche válida.
hora_ingreso_invalida_6 = detectar_horas_invalidas(
    originales_infractores["hora_ingreso"]
)
hora_egreso_invalida_6 = detectar_horas_invalidas(
    originales_infractores["hora_egreso"]
)
hora_invalida_6 = hora_ingreso_invalida_6 | hora_egreso_invalida_6
porcentaje_hora_invalida = 100 * hora_invalida_6.sum() / total_infracciones
print(
    "El porcentaje de infracciones con hora inválida es "
    f"{porcentaje_hora_invalida:.2f}%"
)



- ¿Cuál es el tipo de carga más frecuente en infracciones y qué porcentaje representa?
```
El tipo de carga más frecuente es XXXX con XX.XX%
```


In [ ]:
#@title Ejercicio 06 — Tipo de carga más frecuente en infracciones
conteo_cargas_6 = df_clean["tipo_carga"].value_counts()
if conteo_cargas_6.empty:
    print("No hay tipos de carga informados para calcular la categoría más frecuente.")
else:
    cantidad_carga_maxima = int(conteo_cargas_6.max())
    cargas_mas_frecuentes = conteo_cargas_6[
        conteo_cargas_6.eq(cantidad_carga_maxima)
    ].index
    porcentaje_carga = 100 * cantidad_carga_maxima / total_infracciones
    # Si hay empate, informar todas las categorías con su porcentaje.
    for carga in cargas_mas_frecuentes:
        print(
            f"El tipo de carga más frecuente es {carga} "
            f"con {porcentaje_carga:.2f}%"
        )



- ¿Cuál es el origen más frecuente entre los buques infractores?
```
El origen más frecuente entre infractores es XXXX con XX registros.
```

In [ ]:
#@title Ejercicio 06 — Origen más frecuente entre infractores
conteo_origenes_6 = df_clean["origen"].value_counts()
if conteo_origenes_6.empty:
    print("No hay orígenes informados entre los buques infractores.")
else:
    cantidad_origen_maxima = int(conteo_origenes_6.max())
    origenes_mas_frecuentes = conteo_origenes_6[
        conteo_origenes_6.eq(cantidad_origen_maxima)
    ].index
    # Contar registros: un mismo buque puede aparecer varias veces.
    for origen in origenes_mas_frecuentes:
        print(
            f"El origen más frecuente entre infractores es {origen} "
            f"con {cantidad_origen_maxima} registros."
        )




- ¿Cuál es la duración promedio de estadía en muelle de los buques infractores?
```
La duración promedio de estadía de buques infractores es XX.XX horas.
```

In [ ]:
#@title Ejercicio 06 — Duración promedio de estadía de los infractores
# Usar las duraciones del ejercicio 03, sin reemplazar los ausentes por cero.
duraciones_6 = pd.to_numeric(df_clean["duracion_horas"], errors="coerce")
duracion_promedio_6 = duraciones_6.mean()
if pd.isna(duracion_promedio_6):
    print("No hay duraciones disponibles para calcular la estadía promedio.")
else:
    print(
        "La duración promedio de estadía de buques infractores es "
        f"{duracion_promedio_6:.2f} horas."
    )


## Ejercicio 07

> Puntos: 1

Redactar una conclusión que incluya:

- Evaluación de la calidad del dataset heredado: porcentaje de registros descartados y tipos de error más frecuentes.
- Patrones de infracción detectados: ¿en qué turnos, muelles y tipos de carga se concentran?
- Reflexión sobre el impacto de incorporar estos datos sin limpieza previa al nuevo sistema.
- Al menos una propuesta concreta de mejora para el proceso de captura de datos en el puerto.

Guardar la conclusión en `port_log/reports/conclusion.md`

In [ ]:
#@title Ejercicio 07 — Conclusión y exportación del informe
def describir_frecuencias(conteos: pd.Series, total: int) -> str:
    """Presenta categorías, cantidades y porcentajes sobre el total indicado."""
    return "; ".join(
        f"{categoria}: {int(cantidad)} ({100 * cantidad / total:.2f}%)"
        for categoria, cantidad in conteos.items()
    )


total_original = len(df_raw)
total_descartados = total_original - total_infracciones
porcentaje_descartados = 100 * total_descartados / total_original
# 'filas_antes' conserva la cantidad de registros del ejercicio 03
# antes de filtrar por exceso_velocidad > 0; no cambia en los ejercicios 04 a 06.
total_aptos_antes_filtro = int(filas_antes)
descartados_limpieza = total_original - total_aptos_antes_filtro
excluidos_sin_infraccion = total_aptos_antes_filtro - total_infracciones
if not 0 <= total_infracciones <= total_aptos_antes_filtro <= total_original:
    raise ValueError("Reejecutar en orden: los conteos del ejercicio 03 no coinciden.")

# No sumar estas incidencias: una fila puede tener más de un error.
nulos_originales = df_raw.isna().sum().sort_values(ascending=False)
fechas_originales = df_raw[["fecha_ingreso", "fecha_egreso"]].apply(
    lambda serie: pd.to_datetime(serie, format="mixed", errors="coerce")
)
fechas_invalidas_raw = fechas_originales.isna().any(axis=1)
horas_invalidas_raw = (
    detectar_horas_invalidas(df_raw["hora_ingreso"])
    | detectar_horas_invalidas(df_raw["hora_egreso"])
)
matriculas_normalizadas_7 = df_raw["matricula"].apply(normalizar_matricula)
errores_originales = pd.Series({
    "Algún valor nulo": int(df_raw.isna().any(axis=1).sum()),
    "Alguna fecha inválida o ausente": int(fechas_invalidas_raw.sum()),
    "Alguna hora inválida o ausente": int(horas_invalidas_raw.sum()),
    "Matrícula inutilizable después de normalizar": int(
        matriculas_normalizadas_7.isna().sum()
    ),
}).sort_values(ascending=False)

horas_ingreso_7 = df_clean["hora_ingreso"].str[:2].astype(int)
turnos_7 = pd.cut(
    horas_ingreso_7,
    bins=[0, 6, 12, 18, 24],
    labels=["Madrugada", "Mañana", "Tarde", "Noche"],
    right=False,
)
conteo_turnos_7 = turnos_7.value_counts().sort_values(ascending=False)
conteo_muelles_7 = df_clean["muelle"].value_counts()
turnos_hora_valida_7 = turnos_7.loc[~hora_ingreso_invalida_6]
conteo_turnos_validos_7 = turnos_hora_valida_7.value_counts().sort_values(
    ascending=False
)
detalle_turnos_validos = (
    describir_frecuencias(conteo_turnos_validos_7, len(turnos_hora_valida_7))
    if len(turnos_hora_valida_7)
    else "No hay horas de ingreso válidas para comparar turnos."
)
duraciones_disponibles_7 = int(duraciones_6.notna().sum())
duraciones_con_hora_imputada_7 = int(
    (duraciones_6.notna() & hora_invalida_6).sum()
)
promedio_texto_7 = (
    f"{duracion_promedio_6:.2f} horas"
    if pd.notna(duracion_promedio_6)
    else "no calculable por falta de duraciones"
)

conclusion = f"""# Conclusión — Port Log, Sprint 1

## Calidad del dataset heredado

El archivo original contiene {total_original} movimientos y el dataset limpio
conserva {total_infracciones} registros infractores. Quedaron fuera del conjunto
final {total_descartados} registros ({porcentaje_descartados:.2f}% del original).
Este porcentaje no equivale a un porcentaje de datos erróneos: la limpieza y
el criterio IQR excluyeron {descartados_limpieza} registros
({100 * descartados_limpieza / total_original:.2f}% del original), mientras que
otros {excluidos_sin_infraccion} registros
({100 * excluidos_sin_infraccion / total_original:.2f}%) fueron excluidos por no
superar el límite de velocidad con la tolerancia del 5%. Los valores atípicos
excluidos por IQR son candidatos estadísticos, no errores comprobados por sí solos.

Las incidencias comprobadas en el original, ordenadas por frecuencia, son:
{describir_frecuencias(errores_originales, total_original)}.
Estas categorías se superponen: una misma fila puede presentar más de un problema.
Los nulos por columna se distribuyen así:
{describir_frecuencias(nulos_originales[nulos_originales.gt(0)], total_original)}.
También fue necesario unificar los formatos de fecha y hora y normalizar las
matrículas y los muelles. La normalización de texto no implica necesariamente
que el registro deba descartarse.

Entre las infracciones conservadas, el {porcentaje_fecha_invalida:.2f}% tiene
alguna fecha de ingreso o egreso inválida y el {porcentaje_hora_invalida:.2f}%
tenía alguna hora inválida antes de la imputación. Cada registro se cuenta una
sola vez en cada porcentaje. Una fecha imputada a 1900-01-01 no es una fecha
real y una hora 00:00 no prueba por sí sola un error: puede representar una
medianoche válida. Por eso, la calidad de las horas se recuperó del archivo
original para los mismos movimientos que permanecen en el dataset limpio.

## Patrones de infracción

La distribución por turno, calculada sobre las horas normalizadas del dataset
limpio y consistente con el ejercicio 05, es:
{describir_frecuencias(conteo_turnos_7, total_infracciones)}.
Sin embargo, {int(hora_ingreso_invalida_6.sum())} infracciones tienen la hora de
ingreso imputada a 00:00 y quedan asignadas artificialmente a Madrugada.
Como control, al considerar solo los {len(turnos_hora_valida_7)} registros con
hora de ingreso originalmente válida, la distribución es:
{detalle_turnos_validos}.

Los tres muelles con más registros infractores son:
{describir_frecuencias(conteo_muelles_7.head(3), total_infracciones)}.
Los tres tipos de carga más frecuentes entre las infracciones son:
{describir_frecuencias(conteo_cargas_6.head(3), total_infracciones)}.
Hay {int(df_clean['muelle'].isna().sum())} infracciones sin muelle informado y
{int(df_clean['tipo_carga'].isna().sum())} sin tipo de carga informado; los
porcentajes anteriores usan todas las infracciones como denominador.
Estas concentraciones describen cantidades dentro del conjunto de infractores,
no tasas de riesgo: para comparar propensión a infringir haría falta conocer
el total de movimientos de cada turno, muelle y tipo de carga antes de filtrar.
Tampoco representan cantidades de buques únicos, ya que un buque puede aparecer
en más de un movimiento.

La estadía promedio es {promedio_texto_7}, calculada sobre
{duraciones_disponibles_7} duraciones disponibles y excluyendo
{total_infracciones - duraciones_disponibles_7} valores ausentes. De las
duraciones utilizadas, {duraciones_con_hora_imputada_7} incluyen alguna hora
imputada; por ello el promedio debe interpretarse con esa limitación y no como
una medición completamente verificada de los tiempos de estadía.

## Impacto de migrar sin limpieza

Incorporar directamente el archivo heredado podría asociar movimientos a
matrículas incorrectas, fragmentar un mismo muelle en distintas categorías y
producir estadísticas temporales y duraciones engañosas. Los nulos y los
valores extremos también afectarían las comparaciones de tonelaje y velocidad.
Las imputaciones deben quedar señaladas explícitamente para que el nuevo
sistema no interprete datos sustituidos como observaciones reales. Además,
el dataset final de este sprint contiene solo infracciones: no debe utilizarse
como reemplazo del registro completo de operaciones del puerto.

## Propuesta de mejora

Implementar en el punto de captura un formulario con fechas ISO y horas de
24 horas validadas, matrícula obligatoria con formato definido y un catálogo
único de muelles y tipos de carga. El egreso debe ser posterior al ingreso y
los campos numéricos deben pasar controles de rango definidos con el área
operativa. Los límites de velocidad deben obtenerse del catálogo de muelles.
Los registros que fallen una validación deben ir a una bandeja de revisión,
conservando el valor original, la causa del rechazo y un identificador de
movimiento único. Mantener indicadores separados de dato faltante, inválido
e imputado permitirá corregir errores sin confundirlos con ceros o medianoches
válidas y dejará un historial auditable de la migración.
"""

ruta_conclusion = Path("port_log/reports/conclusion.md")
ruta_conclusion.parent.mkdir(parents=True, exist_ok=True)
ruta_conclusion.write_text(conclusion, encoding="utf-8")
print(conclusion)
print(f"Conclusión guardada en {ruta_conclusion}")


---

In [ ]:
#@title Stage archivos del Sprint 1
%cd /content/Port_Log-Grupo_16

!git add 01_Port_Log-Grupo_16.ipynb
!git add README.md
!git add CHANGELOG.md
!git add port_log/data/raw/port_movements.csv
!git add port_log/data/interim/port_movements.csv
!git add port_log/reports/summary_sprint1.csv
!git add port_log/reports/conclusion.md
!git add port_log/data/interim/plots/top_infractores.jpg
!git add port_log/data/interim/plots/turnos.jpg
!git add port_log/data/interim/plots/meses.jpg
!git add port_log/data/interim/plots/distribucion_exceso.jpg
!git add port_log/data/interim/plots/exceso_por_muelle.jpg
!git add port_log/data/interim/plots/fechas_invalidas.jpg

!git status

# CHECKS

`De aquí en adelante solo ejecutar`

## Visualizar la rama actual

In [ ]:
!ls
!git status

## Visualizar el contenido de cada commit

In [ ]:
!git --no-pager log --name-status Sprint_1